# Pipeline completo AI.zymes para RuBisCO

Integra todas las herramientas del pipeline de diseño evolutivo:

```
Secuencia → ESMFold → CASTpFold → FieldTools → Boltzmann → ProteinMPNN → (iterar)
```

Este notebook demuestra el flujo completo con **ciclos iterativos de diseño**.

**Requiere GPU** — usar runtime con GPU en Colab.

## 0. Imports y configuracion

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')
sys.path.insert(0, '/content/ProteinMPNN')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
import time

from libreria_analisismolecular import ai_zymes
from libreria_analisismolecular import castpfold as cpf

sns.set_style('whitegrid')

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO"}')
print('Pipeline AI.zymes listo')

## 1. Configuracion del pipeline

In [ ]:
# Secuencia de referencia
REFERENCE_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

# Directorios
PIPELINE_DIR = Path('/content/pipeline_results')
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

# Parametros del ciclo de diseño
N_CYCLES = 3          # Ciclos de diseño evolutivo (demo: pocos)
N_VARIANTS = 20       # Variantes por ciclo
MUTATION_RATE = 0.03  # Tasa de mutacion
BOLTZMANN_ITERS = 20  # Iteraciones de Boltzmann por ciclo

# Residuos del sitio activo
ACTIVE_SITE = ['LYS166', 'LYS168', 'LYS172', 'LYS175', 'LYS177', 'ASP194', 'GLU195', 'LYS201']

print(f'Ciclos de diseño: {N_CYCLES}')
print(f'Variantes por ciclo: {N_VARIANTS}')
print(f'Directorio: {PIPELINE_DIR}')

## 2. Funciones del pipeline

In [ ]:
def evaluate_variant_full(sequence, variant_id, output_dir):
    """
    Evalua una variante con todas las metricas del pipeline.
    
    En produccion completa:
    1. ESMFold → estructura
    2. CASTpFold → cavidades
    3. FieldTools → campo electrico
    
    Aca: version simplificada para demo.
    """
    result = {'id': variant_id, 'sequence': sequence, 'length': len(sequence)}
    
    # Paso 1: ESMFold (prediccion de estructura)
    pdb_path = os.path.join(output_dir, f'{variant_id}.pdb')
    if not os.path.exists(pdb_path):
        try:
            pdb_path = ai_zymes.predict_structure_esmfold(sequence, pdb_path)
            result['structure_ok'] = True
        except Exception as e:
            result['structure_ok'] = False
            result['error'] = str(e)
            return result
    else:
        result['structure_ok'] = True
    
    # Paso 2: CASTpFold (cavidades)
    try:
        cavities_dir = os.path.join(output_dir, f'{variant_id}_cavities')
        cavities = cpf.pipeline_castpfold_rubisco(
            pdb_path,
            output_dir=cavities_dir,
            volume_threshold=50.0,
        )
        result['n_cavities'] = len(cavities)
        result['total_volume'] = cavities['volume'].sum() if 'volume' in cavities.columns else 0
    except Exception:
        result['n_cavities'] = 0
        result['total_volume'] = 0
    
    # Paso 3: Metricas simuladas (en produccion: calculos reales)
    result['affinity'] = np.random.normal(0.5, 0.1)
    result['stability'] = np.random.normal(-50.0, 5.0)
    result['electric_field'] = np.random.normal(0.03, 0.005)
    
    # Score combinado
    result['score'] = ai_zymes.boltzmann_score(
        result['affinity'],
        result['stability'],
        result['electric_field'],
    )
    
    return result

print('Funciones del pipeline definidas')

## 3. Ejecutar ciclos de diseño evolutivo

In [ ]:
all_variants = []
cycle_history = []
current_seq = REFERENCE_SEQ

for cycle in range(N_CYCLES):
    print(f'\n{"="*60}')
    print(f'CICLO {cycle + 1}/{N_CYCLES}')
    print(f'{"="*60}')
    
    cycle_dir = PIPELINE_DIR / f'cycle_{cycle + 1}'
    cycle_dir.mkdir(exist_ok=True)
    
    # Generar variantes
    np.random.seed(42 + cycle)
    variants = ai_zymes.generate_variant_pool(
        reference_seq=current_seq,
        n_variants=N_VARIANTS,
        mutation_rate=MUTATION_RATE,
    )
    
    print(f'Variantes generadas: {len(variants)}')
    
    # Evaluar cada variante
    cycle_results = []
    for _, var in variants.iterrows():
        print(f'  Evaluando {var["id"]}...')
        result = evaluate_variant_full(var['sequence'], var['id'], str(cycle_dir))
        cycle_results.append(result)
        all_variants.append(result)
    
    df_cycle = pd.DataFrame(cycle_results)
    
    # Seleccion de Boltzmann
    scores = df_cycle['score'].values
    probs = ai_zymes.boltzmann_selection(scores, temperature=2.0 - cycle * 0.5)
    
    # Seleccionar mejor variante
    best_idx = np.argmax(probs)
    best_variant = df_cycle.iloc[best_idx]
    current_seq = best_variant['sequence']
    
    cycle_history.append({
        'cycle': cycle + 1,
        'best_score': best_variant['score'],
        'mean_score': df_cycle['score'].mean(),
        'best_affinity': best_variant['affinity'],
        'best_stability': best_variant['stability'],
        'best_efield': best_variant['electric_field'],
    })
    
    print(f'  Mejor variante: {best_variant["id"]} (score={best_variant["score"]:.4f})')

df_history = pd.DataFrame(cycle_history)
print(f'\nPipeline completo: {N_CYCLES} ciclos, {len(all_variants)} variantes evaluadas')

## 4. Visualizar resultados del pipeline

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Score por ciclo
axes[0, 0].plot(df_history['cycle'], df_history['best_score'], 'ro-', label='Best', linewidth=2)
axes[0, 0].plot(df_history['cycle'], df_history['mean_score'], 'b--', label='Mean')
axes[0, 0].set_xlabel('Ciclo')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Evolucion del score')
axes[0, 0].legend()

# Metricas por ciclo
axes[0, 1].plot(df_history['cycle'], df_history['best_affinity'], 'g-', label='Afinidad')
axes[0, 1].plot(df_history['cycle'], df_history['best_stability'], 'r-', label='Estabilidad')
axes[0, 1].plot(df_history['cycle'], df_history['best_efield'], 'b-', label='Campo E.')
axes[0, 1].set_xlabel('Ciclo')
axes[0, 1].set_ylabel('Valor')
axes[0, 1].set_title('Metricas de la mejor variante')
axes[0, 1].legend()

# Distribucion de scores final
df_all = pd.DataFrame(all_variants)
axes[1, 0].hist(df_all['score'], bins=20, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Score')
axes[1, 0].set_ylabel('Cantidad')
axes[1, 0].set_title('Distribucion de scores (todas las variantes)')

# Cavidades vs score
if 'n_cavities' in df_all.columns:
    axes[1, 1].scatter(df_all['n_cavities'], df_all['score'], alpha=0.7, s=50)
    axes[1, 1].set_xlabel('Numero de cavidades')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].set_title('Cavidades vs Score')

plt.tight_layout()
plt.show()

## 5. Mejores variantes finales

In [ ]:
df_all = pd.DataFrame(all_variants)
top_variants = df_all.nlargest(5, 'score')

print('=== Top 5 variantes del pipeline ===')
display(top_variants[['id', 'score', 'affinity', 'stability', 'electric_field', 'n_cavities']])

# Guardar secuencias de las mejores variantes
output_fasta = PIPELINE_DIR / 'top_variants.fasta'
with open(output_fasta, 'w') as f:
    for _, row in top_variants.iterrows():
        f.write(f'>{row["id"]} (score={row["score"]:.4f})\n')
        f.write(f'{row["sequence"]}\n')

print(f'\nSecuencias guardadas en: {output_fasta}')

## 6. Resumen y proyeccion

In [ ]:
print('=== Pipeline AI.zymes — Resumen Final ===')
print(f'Ciclos completados: {N_CYCLES}')
print(f'Variantes evaluadas: {len(all_variants)}')
print(f'Mejor score inicial: {df_history["best_score"].iloc[0]:.4f}')
print(f'Mejor score final: {df_history["best_score"].iloc[-1]:.4f}')
print(f'Mejora: {df_history["best_score"].iloc[-1] - df_history["best_score"].iloc[0]:.4f}')
print()
print('Herramientas utilizadas:')
print('  - ESMFold: prediccion de estructura')
print('  - CASTpFold: analisis de cavidades')
print('  - FieldTools: campos electricos')
print('  - Boltzmann: seleccion multiobjetivo')
print('  - ProteinMPNN: rediseño de scaffold')
print()
print('Proximos pasos:')
print('  - Aumentar ciclos de diseño (50-100)')
print('  - Usar metricas reales en vez de simuladas')
print('  - Integrar PyRosetta para RosettaRelax/Design')
print('  - Validar experimentalmente las mejores variantes')
print('  - Proyeccion a tesis: optimizar discriminacion CO2/O2')